In [1]:
%reload_ext autoreload
%autoreload 2


from Stellarator2 import StellaratorTransport
from yancc_wrapper2 import yancc_data
import yancc
import jax.numpy as jnp
import numpy as np
import jax
import MaNTA
import desc
from desc.plotting import plot_comparison
import matplotlib.pyplot as plt
from desc.profiles import SplineProfile
from desc.optimize._constraint_wrappers import ProximalProjection
from desc.objectives import (
    AspectRatio,
    FixBoundaryR,
    FixBoundaryZ,
    FixCurrent,
    FixPsi,
    ForceBalance,
    LinearObjectiveFromUser,
    ObjectiveFunction,
    ObjectiveFromUser,
    RotationalTransform,
    Volume,
)
from desc.grid import Grid, LinearGrid
from desc.geometry import FourierRZToroidalSurface
from desc.equilibrium import Equilibrium, EquilibriaFamily
from desc.backend import tree_unstack
import desc.io
from desc import set_device
from scipy.constants import mu_0
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"




Registering cpu implementation for operation get_solution
Registering gpu implementation for operation get_solution
Registering cpu implementation for operation get_adjoint_gradients
Registering gpu implementation for operation get_adjoint_gradients
Registering cpu implementation for operation run
Registering cpu implementation for operation run_ss
Using cache directory: /global/cfs/cdirs/mp217/eatocco/__pycache__
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [ ]:

st_config = {
    "ParticleSourceCenter": 0.2,
    "ParticleSourceHeight": 5.0,
    "ParticleSourceWidth": 0.2,
    "HeatSourceCenter": 0.2,
    "HeatSourceHeight": 10.0,
    "HeatSourceWidth": 0.4,
    "EdgeTemperature": 0.2,
    "EdgeDensity": 0.4,
    "n0": 0.4,
    "evolveDensity": True,
}
# runner = MaNTA.Runner(st)

rho_upper = 1.0
rtol = 1e-2
atol = 1e-3
# nodes = [0.0,0.5, 0.75, 0.9, 1.0]
npoints = 5
degree = 3
base = 2.0
tau = 10.0
nodes = 1 - 1.0 / np.logspace(1, npoints - 1, base=base, num=npoints - 1)
nodes = np.concatenate(([0], nodes, [1]))
# # %%
solver_config = {
    "OutputFilename": "stellarator_w7x",
    "Polynomial_degree": degree,
    "Grid_points": nodes,
    "tau": tau,
    "Lower_boundary": 0.0,
    "Upper_boundary": rho_upper,
    "Relative_tolerance": rtol,
    "Absolute_tolerance": [atol],
    "delta_t": 1e-4,
    "initialTimestep": 1e-8,
    "MinStepSize": 1e-9,
    "SteadyStateTolerance": 1e-2,
    "restart": False,
    "zeroFlux": True,
}


config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}


points = MaNTA.getNodes(
    nodes,
    solver_config["Polynomial_degree"],
)


yancc_rho = jnp.array(points)
yancc_ntheta = 17
yancc_nzeta = 33

yancc_res = {"na": 43, "nx": 5}
## to allow maximum flexibility to match manta, we use a spline with the same control points as manta \
# + axis and lcfs
# initial pressure is all zeros, can change this if desired
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

eq = desc.examples.get("W7-X")

# Reduce the number of modes (not sure if this is a good thing to do)
# eq.change_resolution(M=4, N=4, L_grid=len(poi 
eq_init = eq.copy()
yancc_wrapper = yancc_data.from_eq(
    points, eq=eq_init, nt=yancc_ntheta, nz=yancc_nzeta, **yancc_res
)
st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)
st.run()



Initializing yancc wrapper
yancc_wrapper initialized successfully with resolution na=43, nx=5.
configuring
Successfully created StellaratorTransport object


ERROR: Residual norm at t = 0: -nan
ERROR: Residual norm at t = 0: inf


In [3]:
from State import State
import equinox as eqx
def make_state(x):
    _var = jnp.array([st.InitialValue(0, x)])
    _der =jnp.array([st.InitialDerivative(0,x)])
    _flux = jnp.zeros(_var.shape)
    return State(_var, _der, _flux, jnp.array([]), jnp.array([]))

def make_vars(i, x):
    _var = st.InitialValue(i, x)
    _der = jax.vmap(st.InitialDerivative, in_axes=(None, 0))(i,x)
    _flux = jnp.zeros(_var.shape)
    return (_var, _der, _flux)

vars = jax.vmap(make_vars, in_axes=(0, None))(jnp.arange(0,3), st.points)
aux = jnp.zeros((len(st.points),))

s0 = State(vars[0].transpose(), vars[1].transpose(), vars[2].transpose(), aux, None).to_manta()

# ind = 4
# s0 = State.from_manta(make_state(st.points[ind]))
# s0 = make_state(st.points[ind])
# s1 = make_state_vec(st.points)
print(s0)
# jac = eqx.filter_jacrev(st.compute_dke_sol)


{'Variable': array([[0.00797004, 0.00239101, 0.00239101],
       [0.06446759, 0.01934028, 0.01934028],
       [0.14297528, 0.04289258, 0.04289258],
       [0.19691962, 0.05907589, 0.05907589],
       [0.20811787, 0.06243536, 0.06243536],
       [0.2344182 , 0.07032546, 0.07032546],
       [0.27112763, 0.08133829, 0.08133829],
       [0.29685525, 0.08905658, 0.08905658],
       [0.30227074, 0.09068122, 0.09068122],
       [0.31509989, 0.09452997, 0.09452997],
       [0.33325705, 0.09997712, 0.09997712],
       [0.34612909, 0.10383873, 0.10383873],
       [0.3497578 , 0.10492734, 0.10492734],
       [0.3626937 , 0.10880811, 0.10880811],
       [0.38111332, 0.114334  , 0.114334  ],
       [0.39425116, 0.11827535, 0.11827535]]), 'Derivative': array([[0.4212128 , 0.12636384, 0.12636384],
       [0.41477122, 0.12443137, 0.12443137],
       [0.40413449, 0.12124035, 0.12124035],
       [0.39379235, 0.1181377 , 0.1181377 ],
       [0.39033073, 0.11709922, 0.11709922],
       [0.38619269, 0.1158

In [4]:
# p1 = jac(s0, st.points[ind], 0.0, st.yancc_wrapper.fields_unstacked[ind], st.vp[ind], st.vpp[ind], st.params)
# print(p1)

In [5]:
physics_data = st.ComputePhysics(s0, st.points, 0.0)

# dphysics_data = st.ComputePhysicsDerivatives(s0, st.points, 0.0)
# print(p1[0].Variable)

In [6]:
print(physics_data)
# print(dphysics_data)

[[Array([ 2.07181954e-06, -6.59868725e-07,  1.43390682e-07,  3.20900649e-07,
       -7.11535559e-07, -2.04447952e-07, -4.51391818e-07, -4.36692061e-08,
       -2.84116795e-08,  2.65562189e-08, -1.63651572e-07, -1.93440411e-07,
        3.84104735e-08, -2.87750568e-07, -3.63398829e-07,  1.18872441e-07],      dtype=float64), Array([ 1.16096225e-06, -3.73470532e-07,  8.32315406e-08,  1.88662892e-07,
       -4.19017991e-07, -1.20728537e-07, -2.67060793e-07, -2.58569398e-08,
       -1.68261344e-08,  1.57362075e-08, -9.70944990e-08, -1.14911308e-07,
        2.28269323e-08, -1.71318552e-07, -2.17130361e-07,  7.12569578e-08],      dtype=float64), Array([ 3.83795107e-08, -1.25656630e-08,  2.85554608e-09,  6.42429237e-09,
       -1.42250632e-08, -4.06212206e-09, -8.83158437e-09, -8.40783909e-10,
       -5.44841743e-10,  5.03978997e-10, -3.05267831e-09, -3.55767469e-09,
        7.03413975e-10, -5.18465755e-09, -6.38179102e-09,  2.04834755e-09],      dtype=float64)], [Array([ 0.35183568,  2.9253951

In [7]:
dphysics_data = st.ComputePhysicsDerivatives(s1, st.points, 0.0)

n=0.499999960655224, Ti=0.20000000000000004, dndrho=-8.270001960440784e-06, dTidrho=0.061104607556236104
n=0.4998298182361687, Ti=0.20000000000000004, dndrho=-0.004410878249604634, dTidrho=-0.0024395948601361847
n=0.4957167608282522, Ti=0.2, dndrho=-0.049564365308652075, dTidrho=0.00024546606655459697
n=0.48394564778035504, Ti=0.2, dndrho=-0.13351648643978284, dTidrho=0.0004040488309864324
n=0.47978148054874664, Ti=0.2, dndrho=-0.1587275517494012, dTidrho=-0.0008496537552538485
n=0.4667095311512487, Ti=0.20000000000000004, dndrho=-0.2307173414591559, dTidrho=-0.00021764350065326332
n=0.43851677187310445, Ti=0.20000000000000004, dndrho=-0.3655171857647545, dTidrho=-0.000406345257005462
n=0.4098042281456502, Ti=0.20000000000000004, dndrho=-0.48722542102668837, dTidrho=-7.593374451537236e-05
n=0.40016868085374235, Ti=0.2, dndrho=-0.5257634751933394, dTidrho=5.80202221256951e-05
n=0.35956060411001023, Ti=0.2, dndrho=-0.679136417692329, dTidrho=-0.0002444852076579693
n=0.2824203407471698, T

In [8]:
print(dphysics_data)

Using cache directory: /global/cfs/cdirs/mp217/eatocco/__pycache__
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]
[[{'Variable': array([[-0.26402498],
       [-0.03240076],
       [-0.01383147],
       [-0.00941071],
       [-0.00878462],
       [-0.00743949],
       [-0.00597198],
       [-0.00513851],
       [-0.00493094],
       [-0.00430829],
       [-0.00369431],
       [-0.0035243 ]]), 'Derivative': array([[0.00504998],
       [0.00501925],
       [0.00493086],
       [0.00483936],
       [0.00481677],
       [0.00476209],
       [0.00469651],
       [0.0046799 ],
       [0.0046836 ],
       [0.00474189],
       [0.00502171],
       [0.00548362]]), 'Flux': array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]]), 'Aux': array([], dtype=float64), 'Scalars': array([], shape=(12, 0), dtype=float64)}], [{'Variable': array([[0.],
       [0.],
       [0.],
     